In [4]:
# This code simulates a single neuron of choice
from brian2 import *
import os
sys.path = [p for p in sys.path if 'Neuron and Synapse Models' not in p and 'Tools' not in p]
os.chdir(os.path.dirname(os.getcwd()))  # Change to the parent directory

import sys
sys.path.append('Neuron and Synapse Models')
from neuronModels import *
sys.path.append('Tools')
from plottingTools import *
from utils import *

from ipywidgets import VBox, HBox, Layout, interactive_output, FloatSlider, FloatText, Checkbox, IntSlider, Label

In [5]:
# Create sliders for parameters with wider layout
control_width = '400px'
desc_width = '200px'

# Basic neuron parameters
timeStep_slider = FloatText(
    min=0.0, 
    max=1.0, 
    step=0.01,
    value=0.5, 
    description='Simulation Time Step:', 
    continuous_update=False, 
    readout_format='.2f',
    style={'description_width': desc_width},
    layout=Layout(width=control_width)
)
neuro_timeStep_slider = FloatText(
    min=0.0, 
    max=1.0, 
    step=0.01,
    value=0.5, 
    description='Neuron Time Step:', 
    continuous_update=False, 
    readout_format='.2f',
    style={'description_width': desc_width},
    layout=Layout(width=control_width)
)
tau_slider = FloatSlider(
    min=0.01, 
    max=10, 
    step=0.01, 
    value=10.0, 
    description='Tau Value:', 
    continuous_update=False,
    style={'description_width': desc_width},
    layout=Layout(width=control_width)
)
amp_slider = FloatSlider(
    min=0.01, 
    max=30, 
    step=0.01, 
    value=1.0, 
    description='Amp Value (mA):', 
    continuous_update=False,
    style={'description_width': desc_width},
    layout=Layout(width=control_width)
)

# Noise parameter slider
sigma_noise_slider = FloatSlider(
    min=0.0,
    max=5.0,
    step=0.1,
    value=0.0,
    description='Noise (sigma):',
    continuous_update=False,
    style={'description_width': desc_width},
    layout=Layout(width=control_width)
)

# F-I curve toggle
show_fi_curve = Checkbox(
    value=False,
    description='Generate F-I Curve',
    indent=False,
    layout=Layout(width=control_width)
)

# Parameters for F-I curve
i_min_slider = FloatSlider(
    min=0.0,
    max=30.0,
    step=0.1,
    value=23.0,
    description='Min Current (mA):',
    continuous_update=False,
    style={'description_width': desc_width},
    layout=Layout(width=control_width)
)

i_max_slider = FloatSlider(
    min=0.1,
    max=100.0,
    step=0.1,
    value=30.0,
    description='Max Current (mA):',
    continuous_update=False,
    style={'description_width': desc_width},
    layout=Layout(width=control_width)
)

i_steps_slider = IntSlider(
    min=5,
    max=50,
    step=1,
    value=50,
    description='Number of Steps:',
    continuous_update=False,
    style={'description_width': desc_width},
    layout=Layout(width=control_width)
)

fi_duration_slider = FloatSlider(
    min=100,
    max=1000,
    step=10,
    value=100,
    description='Simulation Duration (ms):',
    continuous_update=False,
    style={'description_width': desc_width},
    layout=Layout(width=control_width)
)

neurons_per_point_slider = IntSlider(
    min=1,
    max=200,
    step=1,
    value=200,
    description='Neurons per point:',
    continuous_update=False,
    style={'description_width': desc_width},
    layout=Layout(width=control_width)
)

# Group FI curve widgets for conditional display
fi_curve_parameters = VBox([
    Label('F-I Curve Parameters:'),
    i_min_slider,
    i_max_slider,
    i_steps_slider,
    fi_duration_slider,
    neurons_per_point_slider
], layout=Layout(border='1px solid #ddd', padding='10px', margin='10px 0px'))

# Create dictionary for interactive widget
widgets = {
    'timeStep': timeStep_slider,
    'neuroTimeStep': neuro_timeStep_slider,
    'tauVal': tau_slider,
    'ampVal': amp_slider,
    'sigma_noise': sigma_noise_slider,
    'show_fi_curve': show_fi_curve,
    'i_min': i_min_slider,
    'i_max': i_max_slider,
    'i_steps': i_steps_slider,
    'fi_duration': fi_duration_slider,
    'neurons_per_point': neurons_per_point_slider
    }

In [6]:
# Parameters
Vth = -48*mV
V_reset = -80*mV
input = False

In [7]:
def calculate_fi_curve(i_min, i_max, i_steps, duration, timeStep, neuroTimeStep, tauVal, neurons_per_point, sigma_noise=0.0):
    # Set up the current values
    currents = np.linspace(i_min, i_max, i_steps)
    firing_rates_mean = []
    firing_rates_std = []
    
    # Set up the clock and time step
    defaultclock.dt = timeStep*ms
    
    for current in currents:
        # Create a group of neurons with the noisy LIF model
        neuron_eq = Equations(LIF_xi_eq, tau=tauVal*ms, V_rest=-70*mV, sigma_noise=sigma_noise*mV)
        NeuroGrp = NeuronGroup(neurons_per_point, neuron_eq, threshold='V > Vth', reset='V = V_reset', refractory=5*ms,
                               method='euler', dt=neuroTimeStep*ms)
        NeuroGrp.V = -70*mV  # Initial voltage
        NeuroGrp.I_syn = 0.0*amp*ohm
        
        # Set the input current (same for all neurons)
        I_ext = current * mA * ohm
        NeuroGrp.I_ext = I_ext
        
        # Create spike monitor to count spikes
        spike_mon = SpikeMonitor(NeuroGrp)
        
        # Create network and run simulation
        network = Network(NeuroGrp, spike_mon)
        BrianLogger.log_level_error()
        network.run(duration*ms)
        
        # Calculate firing rates using the existing function from utils.py
        rates = compute_firing_rate(spike_mon, neurons_per_point, total_duration=duration*ms)
        rates = compute_firing_rate(spike_mon, neurons_per_point)

        
        # Calculate mean and standard deviation of firing rates
        firing_rates_mean.append(np.mean(rates))
        firing_rates_std.append(np.std(rates))
        
    return currents, np.array(firing_rates_mean), np.array(firing_rates_std)

In [8]:
def interactive_simulator(timeStep, neuroTimeStep, tauVal, ampVal, sigma_noise, show_fi_curve, i_min, i_max, i_steps, fi_duration, neurons_per_point):
    # Clear previous output
    plt.close('all')

    # Always run the single neuron simulation using the noisy model
    defaultclock.dt = timeStep*ms
    
    # Always use the LIF_xi_eq model, with sigma_noise controlling the noise level
    neuron_eq = Equations(LIF_xi_eq, tau=tauVal*ms, V_rest=-70*mV, sigma_noise=sigma_noise*mV)
    NeuroGrp = NeuronGroup(1, neuron_eq, threshold='V > Vth', reset='V = V_reset', method='euler', dt=neuroTimeStep*ms)
    NeuroGrp.V = -70*mV  # Initial voltage
    NeuroGrp.I_syn = 0.0*mA*ohm
    
    # Set noise label based on sigma value
    noise_label = 'Noiseless - ' if sigma_noise == 0 else f'Noisy (σ={sigma_noise:.1f}) - '
    
    M = StateMonitor(NeuroGrp, 'V', record=True)

    I_ext = ampVal * mA * ohm
    NeuroGrp.I_ext = I_ext
    network = Network(NeuroGrp, M)  
    print("Single neuron simulation with I_ext:", I_ext) 
    
    BrianLogger.log_level_error()
    network.run(30*ms)

    # Plot single neuron simulation
    fig1 = plt.figure(figsize=(10, 4))
    plt.plot(M.t/ms, M.V[0]/mV)
    plt.plot([0, 30], [Vth/mV, Vth/mV], 'r--')
    plt.legend(['V', 'Vth'])
    plt.xlabel('Time (ms)')
    plt.ylabel('Voltage (mV)')
    plt.title(f'{noise_label}Single Neuron Simulation')
    plt.tight_layout()
    plt.show()
    
    
    ###########################
    # F-I Curve Simulation
    ###########################
    
    # If F-I curve is enabled, calculate and display it
    if show_fi_curve:
        # Update FI panel visibility (handled in the observer function)
        print("\nCalculating F-I curve...")
        
        # Calculate F-I curve with multiple neurons per point
        currents, firing_rates_mean, firing_rates_std = calculate_fi_curve(i_min, i_max, i_steps, fi_duration, timeStep, neuroTimeStep, tauVal, neurons_per_point, sigma_noise)
        
        # Plot F-I curve with error bars
        fig2 = plt.figure(figsize=(10, 5))
        plt.errorbar(currents, firing_rates_mean, yerr=firing_rates_std, fmt='o-', capsize=5)
        plt.xlabel('Input Current (mA)')
        plt.ylabel('Firing Rate (Hz)')
        noise_title = '' if sigma_noise == 0 else f' (noise σ={sigma_noise:.1f})'
        plt.title(f'F-I Curve{noise_title} (averaged over {neurons_per_point} neurons)')
        plt.grid(True)
        
        has_fits = False
        legend_elements = []
        
        # Linear fit (if there are enough data points)
        if np.max(firing_rates_mean) > 0:  # Only fit if we have some spikes
            non_zero_idx = firing_rates_mean > 0
            if np.sum(non_zero_idx) > 2:  # Need at least 3 points for a reasonable fit
                non_zero_currents = currents[non_zero_idx]
                non_zero_rates = firing_rates_mean[non_zero_idx]
                
                # Linear fit after threshold
                try:
                    fit_coeffs = np.polyfit(non_zero_currents, non_zero_rates, 1)
                    fit_line = np.poly1d(fit_coeffs)
                    
                    # Plot linear fit
                    x_fit = np.linspace(np.min(non_zero_currents), np.max(currents), 100)
                    plt.plot(x_fit, fit_line(x_fit), 'r--', label=f'Linear: {fit_coeffs[0]:.2f}*I + {fit_coeffs[1]:.2f}')
                    has_fits = True
                    legend_elements.append(f'Linear: {fit_coeffs[0]:.2f}*I + {fit_coeffs[1]:.2f}')
                except Exception as e:
                    print(f"Error in linear fit: {e}")
        
                # Rectified Power Law fit
                try:
                    results, best_fit = curveFit_rectPower(non_zero_rates, non_zero_currents, V0=Vth/mV)
                    
                    print(results.fit_report())
                    
                    # Print parameters with uncertainties
                    print("\nPower Law Fit Parameters:")
                    
                    # Check if results has the standard lmfit structure
                    if hasattr(results, 'params'):
                        # Standard lmfit ModelResult approach
                        for param_name in results.params:
                            param = results.params[param_name]
                            if hasattr(param, 'stderr') and param.stderr is not None:
                                print(f"{param_name} = {param.value:.4f} ± {param.stderr:.4f}")
                            else:
                                print(f"{param_name} = {param.value:.4f} (uncertainty not available)")

                    # Generate points for power law curve
                    x_dense = np.linspace(np.min(non_zero_currents), np.max(non_zero_currents), 200)
                    y_fit = rect_power(x_dense, best_fit['a'], best_fit['V0'], best_fit['p'])
                    
                    # Plot power law fit with different color/style
                    plt.plot(x_dense, y_fit, 'g-.', linewidth=2)
                    power_label = f"Power: a={best_fit['a']:.2f}, V0={best_fit['V0']:.2f}, p={best_fit['p']:.2f}"
                    legend_elements.append(power_label)
                    has_fits = True
                except Exception as e:
                    print(f"Error in power law fit: {e}")
        
        # Add legend if we have any fits
        if has_fits:
            plt.legend(legend_elements)
        
        plt.tight_layout()
        # plt.show()

In [9]:
# Define a function to update FI parameters visibility based on checkbox
def update_fi_visibility(change):
    if change['new']:  # If checkbox is checked
        fi_curve_parameters.layout.display = 'flex'
    else:
        fi_curve_parameters.layout.display = 'none'

# Register the observer
show_fi_curve.observe(update_fi_visibility, names='value')

# Initial state - hide FI parameters if checkbox is unchecked
if not show_fi_curve.value:
    fi_curve_parameters.layout.display = 'none'

# Create the interactive output
out = interactive_output(interactive_simulator, widgets)

# Group basic parameters
basic_parameters = VBox([
    Label('Basic Parameters:'),
    timeStep_slider, 
    neuro_timeStep_slider, 
    tau_slider, 
    amp_slider,
    sigma_noise_slider
], layout=Layout(border='1px solid #ddd', padding='10px', margin='10px 20px 10px 0px'))

all_fi_curve_parameters = VBox([
    show_fi_curve,
    fi_curve_parameters
], layout=Layout(border='1px solid #ddd', padding='10px', margin='10px 0px'))

# Create control panel with all widgets
controls = HBox([
    basic_parameters,
    all_fi_curve_parameters
], layout=Layout(margin='20px 0 0 0', justify_content='space-around'))

# Display layout with output above controls
display(VBox([
    out,  # Output area (plots) at the top
    controls  # Controls below the plots
], layout=Layout(align_items='center')))